# 042 — Convert selected records to JSON

Reads the combined ESM and NGA-Sub conversion lists from `040` and writes one
per-record JSON file per selected `(record_identifier, component)` into
`cfg["proc_data"]["gm_records"]`, giving both databases a single, uniform
on-disk schema for the downstream MSA runs (`050`). Records are pulled from the
(network-mounted) raw archives in parallel; rows whose JSON already exists are
skipped, so the notebook is cheap and safe to re-run as more records arrive.

The conversion logic lives in two importable command-line scripts in
`phd_project/scripts/data_handling/` — `convert_esm_records_to_json.py`
(ASDF/HDF5 → JSON) and `convert_ngasub_records_to_json.py` (AT2 → JSON). Here we
call their `convert_*_records()` run-functions; see **Command-line usage** below
to run them standalone.

**Upstream** (from `040`, in `cfg["proc_data"]["gm_selection"]`):
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv` — combined
  `(record_identifier, component)` conversion lists across both AvgSA campaigns.

Also reads the raw record archives on the share
(`cfg["raw_data"]["esm_hdf5_folder"]`, `cfg["raw_data"]["ngasub_folder"]`) and,
for NGA-Sub, the reference flatfiles `NGASub_SA_H1_H2_filenamemap.csv`
(H1/H2 → physical channel) and `NGASub_Metadata_SA_rotD50.csv`
(event/station codes) in `cfg["raw_data"]["gm_flatfiles"]`.

**Output** (in `cfg["proc_data"]["gm_records"]`):
- One `{record_identifier}__{component}.json` per converted record, sharing a
  common schema (`eq_index`, `database`, `dt`, `duration`, `units`,
  `record_type`, `component`, event/station codes, and the acceleration
  `record` in g, last).

**Run order** — run top to bottom after `040` has (re)written the conversion
lists. Records still missing from the raw archives are reported as skipped for
inspection rather than aborting the run.

In [1]:
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.data_handling.convert_esm_records_to_json import (
    convert_esm_records,
)
from phd_project.scripts.data_handling.convert_ngasub_records_to_json import (
    convert_ngasub_records,
)

cfg = config.load_config()

## Paths

In [2]:
# Source record archives (network share - external resource)
ESM_SRC = cfg["raw_data"]["esm_hdf5_folder"]
NGASUB_SRC = cfg["raw_data"]["ngasub_folder"]

# Combined selection lists (from wp1pt3pt8i)
esm_selection_csv = cfg["proc_data"]["gm_selection"] / "esm_records_to_convert.csv"
ngasub_selection_csv = cfg["proc_data"]["gm_selection"] / "ngasub_records_to_convert.csv"

# NGA-Sub reference flatfiles
ngasub_filename_map = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H1_H2_filenamemap.csv"
ngasub_metadata = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"

# Output folder for the converted JSON records
output_dir = cfg["proc_data"]["gm_records"]

## Convert ESM records

In [3]:
n_esm, esm_skipped = convert_esm_records(
    ESM_SRC, esm_selection_csv, output_dir)


Done: 0 converted, 1848 already present (skipped), 0 failed (max_workers=125).


## Convert NGA-Sub records

In [4]:
n_ngasub, ngasub_skipped = convert_ngasub_records(
    NGASUB_SRC, ngasub_selection_csv, output_dir,
    filename_map=ngasub_filename_map, metadata=ngasub_metadata)

⚠️  RSN 7001179 H2: AT2 file for 'TAP094--E.AT2' not found — skipped.⚠️  RSN 4032428 H2: AT2 file for 'MYG004-NS.AT2' not found — skipped.

⚠️  RSN 4006380 H1: AT2 file for 'FKSH16EW2.AT2' not found — skipped.
⚠️  RSN 4018859 H2: AT2 file for 'AKT013NS.AT2' not found — skipped.
⚠️  RSN 4032414 H2: AT2 file for 'IWT016-NS.AT2' not found — skipped.
⚠️  RSN 4018131 H1: AT2 file for 'TCG007-EW.AT2' not found — skipped.
⚠️  RSN 4040879 H1: AT2 file for 'YMN010-EW.AT2' not found — skipped.
⚠️  RSN 2000773 H1: AT2 file for 'B033EH1.AT2' not found — skipped.
⚠️  RSN 7005809 H1: AT2 file for 'TCU120--N.AT2' not found — skipped.
⚠️  RSN 4009643 H1: AT2 file for 'NGNH33EW2.AT2' not found — skipped.
⚠️  RSN 4027472 H2: AT2 file for 'MYZ012-NS.AT2' not found — skipped.
⚠️  RSN 2000811 H2: AT2 file for 'L02DBHE.AT2' not found — skipped.
⚠️  RSN 2001400 H1: AT2 file for 'HSOENN.AT2' not found — skipped.
⚠️  RSN 2001550 H2: AT2 file for 'MODHHE.AT2' not found — skipped.
⚠️  RSN 4008143 H2: AT2 file fo

## Records that could not be converted

Any selected records that were skipped (e.g. missing source file) are listed
below for inspection.

In [5]:
esm_skipped_df = pd.DataFrame(esm_skipped, columns=["record_identifier", "component", "reason"])
ngasub_skipped_df = pd.DataFrame(ngasub_skipped, columns=["record_identifier", "component", "reason"])

print(f"ESM skipped: {len(esm_skipped_df)} | NGASub skipped: {len(ngasub_skipped_df)}")
display(esm_skipped_df)
display(ngasub_skipped_df)

ESM skipped: 0 | NGASub skipped: 633


,record_identifier,component,reason


,record_identifier,component,reason
0,7001179,H2,AT2 file not found
1,4032428,H2,AT2 file not found
2,4006380,H1,AT2 file not found
3,4018859,H2,AT2 file not found
4,4032414,H2,AT2 file not found
...,...,...,...
628,4022520,H2,AT2 file not found
629,4028069,H1,AT2 file not found
630,7006064,H1,AT2 file not found
631,4022463,H1,AT2 file not found
